# Week 2, day 5 (morning) — Worksheet 03 SOLUTIONS: comprehensions   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q6 contradicts a slide, and Q10 contradicts what most people assume. Both are
noted where they happen.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — Comprehensions. Run this once.
temps_c = [0, 12, 18, 23, 31, 7]
words = ["Extract", "transform", "LOAD", "validate"]

inventory = {"widget": 12, "gizmo": 0, "doohickey": 7, "sprocket": 0}
readings = [("s1", 4.2), ("s2", 0.0), ("s3", 7.9), ("s4", 3.1)]

on_sale_items = ["Chopped tomatoes", "Banana", "Red kidney beans"]
teams = [["ana", "bo"], ["cai", "dee", "eve"], ["fay"]]

# name -> the skills they listed. The skills are a LIST. Remember that for Q11.
tagged = [("ana", ["sql", "python"]), ("bo", ["go"]), ("cai", ["sql"])]

print("temps_c:  ", temps_c)
print("words:    ", words)
print("inventory:", inventory)

PART A — The same loop, written twice

### Question 1

Loop, then comprehension. -> `[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]` twice, then `True`.

Four lines become one, and the one line is not a different technique — it is
the same loop with the pieces rearranged. `[2 * i for i in range(10)]`
holds the loop header (`for i in range(10)`) and the thing you were
appending (`2 * i`), in the opposite order to how you wrote them.

The brackets decide what you get back. Square brackets build a list; curly
braces with a `key: value` build a dict, and curly braces without build a
set. Both are on this sheet.

A comprehension always produces a **new** collection. It never modifies the
one it reads — which is exactly the fix for worksheet 02's Q10.

In [ ]:
double_value = []
for i in range(10):
    double_value.append(2 * i)
print(double_value)

double_comp = [2 * i for i in range(10)]
print(double_comp)

print(double_value == double_comp)

### Question 2

Two plain conversions. -> `[32.0, 53.6, 64.4, 73.4, 87.8, 44.6]` and `[7, 9, 4, 8]`. Both `6 -> 6` and `4 -> 4`.

With no `if` anywhere, a comprehension is a one-to-one transformation: six
values in, six values out. That is worth noticing now, because Q5 turns on
exactly this property.

`0` Celsius came out as `32.0`, a float, not `32`. The `/ 5` did that —
division with `/` always produces a float in Python 3, even when it divides
evenly. `//` is the one that keeps integers.

In [ ]:
temps_f = [c * 9 / 5 + 32 for c in temps_c]
print(temps_f)

lengths = [len(w) for w in words]
print(lengths)

print(len(temps_c), "->", len(temps_f))
print(len(words), "->", len(lengths))

PART B — The two places `if` can go

### Question 3

The filter form. -> `[18, 23, 31] 3 of 6`, and `['Extract', 'LOAD'] 2 of 4`.

`if` at the **end** is a gate: each item is tested, and only the ones that
pass are put in the list. Six temperatures in, three out.

There is no `else`, and there cannot be one. If an item fails the test
nothing goes into the list for it — the list is simply shorter.

`w != w.lower()` is a neat trick for "has at least one capital in it":
lowercasing changes the string only if there was something to change.

In [ ]:
warm = [t for t in temps_c if t > 15]
print(warm, len(warm), "of", len(temps_c))

loud = [w for w in words if w != w.lower()]
print(loud, len(loud), "of", len(words))

### Question 4

The map form, including the deck's example. -> `[0.5, 4, 1.5, 16, 2.5] 5`, then `['cold', 'cold', 'warm', 'warm', 'warm', 'cold'] 6`.

`if` at the **front** is a choice of value, not a gate. Every item produces
something, so five in gives five out and six in gives six out.

This is the conditional expression from worksheet 01 Q9 —
`<expr1> if <cond> else <expr2>` — dropped inside a comprehension. It is
the same construct, and it needs its `else` for the same reason: it has to
evaluate to a value whichever way the test goes.

The odd numbers came back as `0.5`, `1.5`, `2.5` — floats from `/`, while
the even ones stayed ints from `**`. One list, two types. Python allows
it; whatever reads that list next may not.

In [ ]:
deck = [e ** 2 if e % 2 == 0 else e / 2 for e in [1, 2, 3, 4, 5]]
print(deck, len(deck))

banded = ["warm" if t > 15 else "cold" for t in temps_c]
print(banded, len(banded))

### Question 5

The two forms on the same data. -> filter `[18, 23, 31] 3`; map `[0, 0, 18, 23, 31, 0] 6`. **Same test, same input, different lengths.**

This is the distinction to carry out of the sheet:

| | position | `else` | length of result |
|---|---|---|---|
| **filter** | `if` at the end | not allowed | can shrink |
| **map** | `if` at the front | required | always the same |

The filter form answers *which items do I want*. The map form answers
*what should each item become*. Reach for the wrong one and nothing
raises — you get six rows where you wanted three, with the rejects sitting
there as zeros, and a `sum()` over them still works.

You can use both at once: `[t if t > 25 else 0 for t in temps_c if t > 15]`
filters first, then maps what survived. Correct, and about as much as one
line should be asked to carry.

In [ ]:
filtered = [t for t in temps_c if t > 15]           # if at the END
mapped = [t if t > 15 else 0 for t in temps_c]      # if at the FRONT

print("filter:", filtered, len(filtered))
print("map:   ", mapped, len(mapped))

# The FILTER form (if at the end) decides whether an item is kept, so the
# result can be shorter -- 3 items out of 6 here. It takes no else: there is
# nothing to put in the list for an item you are dropping.
#
# The MAP form (if at the front) decides WHICH VALUE each item becomes, so
# the result is always the same length as the input. It REQUIRES else,
# because every item has to turn into something.

PART C — Dictionaries, sets, and pairs

### Question 6

A list turned into a dictionary. -> `{'Chopped tomatoes': 'On sale', 'Banana': 'On sale', 'Red kidney beans': 'On sale'}`. **The slide prints `'Out of stock'` for all three.**

The slide's code says `"On sale"` and the output underneath it says
`'Out of stock'`. There is no route from one to the other: a comprehension
puts in exactly the expression you wrote, three times over. The output was
pasted from an earlier version of the cell.

Worth knowing as a pattern in its own right — this is how you give every
key the same starting value, which is how you initialise a counter or a
flag table before filling it in.

In [ ]:
on_sale_dict = {item: "On sale" for item in on_sale_items}
print(on_sale_dict)

# The slide's code says "On sale" and its printed output says "Out of stock".
# A comprehension puts in exactly the expression you wrote -- there is no
# route from one to the other. The slide's output was pasted from an
# earlier version of the cell.

### Question 7

Three dicts from one. -> `{'widget': 24, 'gizmo': 0, 'doohickey': 14, 'sprocket': 0}`; `{'gizmo': 0, 'sprocket': 0} 2`; and `by_count` is `{12: 'widget', 0: 'sprocket', 7: 'doohickey'}` — **3 entries from 4 items.**

`doubled` and `out_of_stock` are the map and filter forms again, over
`.items()` this time, with two loop variables unpacked exactly as in
worksheet 02 Q8.

`by_count` is the one to look at. `gizmo` and `sprocket` both have a count
of `0`, and a dictionary cannot hold the same key twice — so the second one
overwrote the first and `gizmo` is gone. Four items in, three out, no
error, no warning.

**Inverting a dictionary is only safe when the values are unique**, and
yours usually are not. Note also which one survived: the *last* one
written wins, so the answer depends on the order the dictionary was built
in. That is a bug that reproduces differently depending on the input file.

In [ ]:
doubled = {k: v * 2 for k, v in inventory.items()}
print(doubled)

out_of_stock = {k: v for k, v in inventory.items() if v == 0}
print(out_of_stock, len(out_of_stock))

by_count = {v: k for k, v in inventory.items()}
print(by_count, len(by_count))

### Question 8

Set, dict, and `enumerate`. -> a set of the four sensor ids (length `4`); `{'s1': 4.2, 's2': 0.0, 's3': 7.9, 's4': 3.1}`; and `['1. Extract', '2. transform', '3. LOAD', '4. validate']`.

**The set will print in a different order for you, and in a different order
again next time.** Compare the contents and the length, never the
arrangement.

Curly braces with a bare expression build a **set**; add a colon and they
build a **dict**. Same brackets, and the colon is the entire difference —
which is also why `{}` is an empty dict and never an empty set.

`enumerate(words, start=1)` is how you get 1-based numbering without doing
`i + 1` arithmetic in the body. The deck does not mention `start`; it is
the most useful thing about `enumerate` after the index itself.

In [ ]:
sensor_ids = {sid for sid, value in readings}
print(sensor_ids, len(sensor_ids))

readings_dict = {sid: value for sid, value in readings}
print(readings_dict)

numbered = [f"{i}. {w}" for i, w in enumerate(words, start=1)]
print(numbered)

PART D — Nesting, scope, and where to stop

### Question 9

Flattening with a nested comprehension. -> `['ana', 'bo', 'cai', 'dee', 'eve', 'fay'] 6`, then `True`.

Two `for` clauses in one comprehension, and they read **left to right in
the same order you would stack the loops top to bottom**: outer first,
inner second. If you write them the other way round you get `NameError`,
because the inner clause is looping over something the outer one has not
produced yet.

That is the one rule worth memorising here, because it looks backwards.
The *expression* comes first and the loops come after, so people expect the
loops to be reversed too. They are not.

The loop version is six lines and does the same job. At two levels the
comprehension is still readable. At three it is not — write the loops.

In [ ]:
flat = [name for team in teams for name in team]
print(flat, len(flat))

flat_loop = []
for team in teams:
    for name in team:
        flat_loop.append(name)

print(flat_loop == flat)

### Question 10

Loop-variable scope. -> `[0, 1, 2]`, then **`untouched`**, then **`2`**.

The two constructs behave differently and this catches almost everyone.

A comprehension has **its own scope**. Its loop variable is created inside
the brackets and destroyed when they close, so the outer `x` never moved —
even though the comprehension used the same name.

A `for` loop does **not**. Its loop variable is an ordinary variable in the
surrounding code, it survives the loop, and it holds whatever the last pass
left in it — `2` here. Reuse a name and you overwrite it silently.

(In Python 2 comprehensions leaked too. This was changed deliberately, and
it is one of the few places where the newer behaviour is strictly safer.)

In [ ]:
x = "untouched"

squares = [x for x in range(3)]
print(squares)
print(x)          # the comprehension's x lived and died inside the brackets

for x in range(3):
    pass

print(x)          # the for loop's x is a normal variable, and it is still here

### Question 11

A list as a dictionary key. -> the good dictionary prints, then `TypeError: unhashable type: 'list'`.

Keyed by name it works — a string is immutable, so its hash is stable and
the dictionary can always find the entry again.

A list is not. It can be changed after it is used as a key, at which point
its hash would no longer match where the entry was filed, and the entry
would be unreachable. Rather than let that happen, Python refuses lists as
keys outright.

`tuple(skills)` gives you an immutable version that *is* allowed. But stop
and ask first — a compound key made of a whole skill list is almost never
what you want. Keying by name and searching the values usually is.

In [ ]:
# Keyed by the name -- fine, a string is hashable.
by_name = {name: skills for name, skills in tagged}
print(by_name)

# This is SUPPOSED to raise. A dictionary key must be hashable, and a list
# is not: it can be changed after you use it, which would strand the entry.
# A tuple of the same skills would work -- tuple(skills) -- or key it by
# name and search the values instead.
by_skills = {skills: name for name, skills in tagged}
print(by_skills)